In [ ]:
import os
os.chdir(path_to_wd)
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import snapatac2 as snap
import anndata as ad
import pandas as pd
import scanpy as sc
import scvi
import numpy as np
import polars as pl
import pickle
from scipy.stats import zscore
import json
import pycats
from pandas.api.types import CategoricalDtype

In [ ]:
# Input files
fragment_file = "Reproducibility/Data/ATAC_fragments/atac_fragments_Global.tsv.gz"

# Preprocessing
data = snap.pp.import_data(
     fragment_file,
     chrom_sizes=snap.genome.hg38,
     min_num_fragments=200,
     file="Reproducibility/Data/UC_DOGMA_snapatac2_global.h5ad",  # Optional
     sorted_by_barcode=False,
     tempdir="atac_fragments/tmp",
     # n_jobs=1
     )

snap.pl.frag_size_distr(data)

fig = snap.pl.frag_size_distr(data, show=False)
fig.update_yaxes(type="log")
fig.show()

snap.metrics.tsse(data, snap.genome.hg38)

snap.pl.tsse(data)
snap.pp.add_tile_matrix(data)
data.close()

In [ ]:
atac_full = snap.read("Reproducibility/Data/UC_DOGMA_snapatac2_global.h5ad", backed=None)

In [ ]:
def run_snapatac_peak_calling(atac_full, lineages, dir):
    for lineage in lineages:
        print(f"Processing lineage: {lineage}")
        save_dir=f'{dir}{lineage}/'
        if lineage == "Malignant":
            atac = atac_full[(atac_full.obs["lineage"] == "Epithelial") & (~atac_full.obs["celltype"].isin(["Normal"]))].copy()
        else:
            atac = atac_full[atac_full.obs["lineage"] == lineage].copy()
        snap.tl.macs3(atac, groupby="celltype", n_jobs=16)
        peaks = snap.tl.merge_peaks(atac.uns["macs3"], snap.genome.hg38)
        peak_mat = snap.pp.make_peak_matrix(atac, use_rep=peaks["Peaks"])
        peak_mat.write(f"{save_dir}snapatac2_{lineage}_peak_mat.h5ad")
        peak_mat.var.index.to_frame(index=False).to_csv(
            f"{save_dir}snapatac2_{lineage}_peak_mat_index.txt", index=False, header=False
        )

In [ ]:
lineages = ["Malignant", "CD4_T", "CD8_T_NK_ILC", "B", "Myeloid", "MSC"]

run_snapatac_peak_calling(atac_full = atac_full, lineages = lineages, dir="Reproducibility/Results/SnapATAC2/")

In [ ]:
############################
# BCG pre/post
############################
status = 'BCG'
save_dir=f'Reproducibility/Results/SnapATAC2/{status}/'

atac = atac_full[atac_full.obs["sample"].isin(["BC_011","BC_016","BC_023","BC_033","BC_037",
                                               "BC_027","BC_032","BC_039","BC_040","BC_043","BC_044","BC_047","BC_048"])].copy()
atac = atac[atac.obs["lineage"].isin(["CD4_T","CD8_T_NK_ILC","B","Myeloid"])]
atac.obs["BCG"] = np.select(
    [   atac.obs["sample"].isin(["BC_011","BC_016","BC_023","BC_033","BC_037"]),
        atac.obs["sample"].isin(["BC_027","BC_032","BC_039","BC_040","BC_043","BC_044","BC_047","BC_048"])
    ],
    ["pre", "post"],
    default="unknown"  # optional, for samples not in either list
)
atac.obs['coarse_celltype_w_BCG'] = atac.obs['coarse_celltype'].astype(str) + '_' + atac.obs['BCG'].astype(str)

snap.tl.macs3(atac, groupby='coarse_celltype_w_BCG', n_jobs=16)

peaks = snap.tl.merge_peaks(atac.uns['macs3'], snap.genome.hg38)
peak_mat = snap.pp.make_peak_matrix(atac, use_rep=peaks['Peaks'])
peak_mat.write(f"{save_dir}snapatac2_{status}_peak_mat.h5ad")
peak_mat_index_df = peak_mat.var.index.to_frame(index=False)
peak_mat_index_df.to_csv(f"{save_dir}snapatac2_{status}_peak_mat_index.txt", index=False, header=False)

In [ ]:
# Regression-based differential test
cell_types = ["Mono_Mac", "Treg", "NK_ILC", "DC", "CD8_T", "CD4_Tconv", "B"]

for cell_type in cell_types:
    group1 = f"{cell_type}_post"
    group2 = f"{cell_type}_pre"
    cell_group1 = atac.obs['coarse_celltype_w_BCG'] == group1
    cell_group2 = atac.obs['coarse_celltype_w_BCG'] == group2
    peaks_selected = np.logical_or(
        peaks[group1].to_numpy(),
        peaks[group2].to_numpy(),
    )

    diff_peaks = snap.tl.diff_test(
        peak_mat,
        cell_group1=cell_group1,
        cell_group2=cell_group2,
        features=peaks_selected,
        min_log_fc=0.1,
        min_pct=0
    )
    
    columns = ['peak','log2FC','p_value','p_value_adj']
    tmp_df = pd.DataFrame(data=diff_peaks, columns=columns)
    tmp_df = tmp_df.set_index('peak')

    tmp_path = f'{save_dir}snapatac2_BCG_diff_peaks_{cell_type}.txt'
    tmp_df.to_csv(tmp_path, sep='\t')

In [ ]:
############################
# Malignant
############################
# Finding marker regions
save_dir='Reproducibility/Results/SnapATAC2/Malignant/'
peak_mat = sc.read(f"{save_dir}snapatac2_Malignant_peak_mat.h5ad")
marker_peaks = snap.tl.marker_regions(peak_mat, groupby='celltype', pvalue=0.05)

# Convert each Index in the dictionary to a set for easier operations
marker_sets = {key: set(values) for key, values in marker_peaks.items()}
all_values = set.union(*marker_sets.values())
common_values = set.intersection(*marker_sets.values())
filtered_marker_peaks = {
    key: values - common_values for key, values in marker_sets.items()
}
filtered_marker_peaks = {
    key: pd.Index(list(values)) for key, values in filtered_marker_peaks.items()
}

filtered_marker_peaks_list = {key: value.tolist() for key, value in filtered_marker_peaks.items()}
with open(f"{save_dir}snapatac2_Malignant_marker_peaks_filtered_p_0.05.json", 'w') as json_file:
    json.dump(filtered_marker_peaks_list, json_file)

filtered_marker_peaks_values = np.concatenate([[x for x in p] for p in filtered_marker_peaks.values()])
n = len(filtered_marker_peaks_values)

new_order = ['LUM', 'NRP', 'SQM', 'MES', 'NEC']
count_tmp = snap.tl.aggregate_X(peak_mat, groupby='celltype', normalize="RPKM")
count = count_tmp[new_order]
names = count.obs_names
count = pl.DataFrame(count.X.T)
count.columns = list(names)

idx_map = {x: i for i, x in enumerate(peak_mat.var_names)}
idx = [idx_map[x] for x in filtered_marker_peaks_values]
mat = np.log2(1 + count.to_numpy()[idx, :])
tmp = pd.DataFrame(mat, index = filtered_marker_peaks_values, columns = count.columns)
duplicated_rows = tmp.index.duplicated(keep=False)
print(tmp[duplicated_rows])

tmp2 = tmp[~tmp.index.duplicated(keep='first')].copy()
tmp2_zscore = tmp2.apply(zscore, axis=1)

tmp2_zscore.to_csv(f"{save_dir}snapatac2_Malignant_filtered_marker_peak_zscored.txt", sep='\t')